<a href="https://colab.research.google.com/github/Piyush-Sharma-0/Customer_segmentation_project/blob/main/Final_CBSOT_Customer_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn Prediction & Segmentation


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## Data Loading


In [ ]:
try:
    BASE_PATH = Path(__file__).resolve().parent
except NameError:
    BASE_PATH = Path.cwd()

DATA_PATH = BASE_PATH / "Telco_customer_churn.xlsx"

df = pd.read_excel(DATA_PATH)
print(f"Data loaded successfully. Shape: {df.shape}")
print("\nDataset information:")
df.info()


## Data Cleaning


In [ ]:
# Convert total charges to numeric and treat missing values for new customers as zero.
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df["Total Charges"] = df["Total Charges"].fillna(0)

df_clean = df.copy()

print(f"\nMissing values in 'Total Charges' after cleaning: {df_clean['Total Charges'].isnull().sum()}")


## EDA


In [ ]:
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.boxplot(x="Churn Label", y="Tenure Months", hue="Churn Label", data=df_clean, palette="viridis", legend=False)
plt.title("Tenure Distribution by Churn")

plt.subplot(1, 3, 2)
sns.kdeplot(
    df_clean[df_clean["Churn Value"] == 1]["Monthly Charges"],
    label="Churn",
    fill=True,
    color="red",
)
sns.kdeplot(
    df_clean[df_clean["Churn Value"] == 0]["Monthly Charges"],
    label="No Churn",
    fill=True,
    color="blue",
)
plt.title("Monthly Charges Density")
plt.legend()

plt.subplot(1, 3, 3)
contract_churn = pd.crosstab(df_clean["Contract"], df_clean["Churn Label"], normalize="index")
contract_churn.plot(kind="bar", stacked=True, ax=plt.gca())
plt.title("Contract Type Churn Rate")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(df_clean["Tenure Months"], bins=30, kde=True)
plt.xlabel("Tenure Months")
plt.ylabel("Customer Count")
plt.title("Distribution of Tenure Months")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x="Churn Label", y="Tenure Months", data=df_clean)
plt.xlabel("Churn Label")
plt.ylabel("Tenure Months")
plt.title("Churn vs Tenure Months")
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(df_clean["Monthly Charges"], bins=30, kde=True)
plt.xlabel("Monthly Charges")
plt.ylabel("Customer Count")
plt.title("Distribution of Monthly Charges")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x="Churn Label", y="Monthly Charges", data=df_clean)
plt.xlabel("Churn Label")
plt.ylabel("Monthly Charges")
plt.title("Churn vs Monthly Charges")
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(x="Contract", hue="Churn Label", data=df_clean)
plt.xlabel("Contract")
plt.ylabel("Customer Count")
plt.title("Contract vs Churn")
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(x="Internet Service", hue="Churn Label", data=df_clean)
plt.xlabel("Internet Service")
plt.ylabel("Customer Count")
plt.title("Internet Service vs Churn")
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(x="Payment Method", hue="Churn Label", data=df_clean)
plt.xlabel("Payment Method")
plt.ylabel("Customer Count")
plt.title("Payment Method vs Churn")
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(x="Tech Support", hue="Churn Label", data=df_clean)
plt.xlabel("Tech Support")
plt.ylabel("Customer Count")
plt.title("Tech Support vs Churn")
plt.show()

avg_tenure = df_clean.groupby("Churn Label")["Tenure Months"].mean()
print("\nAverage tenure by churn label:")
print(avg_tenure)

monthly_charge_summary = df_clean["Monthly Charges"].describe()
print("\nMonthly charges summary:")
print(monthly_charge_summary)

monthly_charges_churn_yes = df_clean[df_clean["Churn Label"] == "Yes"]["Monthly Charges"].quantile([0.25, 0.5, 0.75])
monthly_charges_churn_no = df_clean[df_clean["Churn Label"] == "No"]["Monthly Charges"].quantile([0.25, 0.5, 0.75])
print("\nMonthly charge quantiles for churned customers:")
print(monthly_charges_churn_yes)
print("\nMonthly charge quantiles for retained customers:")
print(monthly_charges_churn_no)

numerical_cols = ["Tenure Months", "Monthly Charges", "Churn Value", "Churn Score", "CLTV"]
correlation_matrix = df_clean[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

print("\nContract churn distribution:")
print(contract_churn)


## Feature Engineering


In [ ]:
drop_columns = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",
    "Churn Label",
    "Churn Score",
    "Churn Reason",
    "CLTV",
]

df_model = df_clean.drop(columns=drop_columns)
df_encoded = pd.get_dummies(df_model, drop_first=True)

X = df_encoded.drop("Churn Value", axis=1)
y = df_encoded["Churn Value"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")


## Modeling


In [ ]:
# Train the selected final Random Forest configuration from the original notebook.
rf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
)

rf_final.fit(X_train, y_train)

y_pred = rf_final.predict(X_test)
y_prob = rf_final.predict_proba(X_test)[:, 1]

print("\n--- Final Model Evaluation ---")
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

feature_importances = pd.Series(rf_final.feature_importances_, index=X.columns)
plt.figure(figsize=(10, 6))
feature_importances.nlargest(10).sort_values().plot(kind="barh", color="teal")
plt.title("Top 10 Business Drivers of Churn")
plt.xlabel("Relative Importance")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.show()


## Model Performance Reporting
This section adds presentation-ready evaluation outputs without changing the selected Random Forest pipeline.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_curve,
)

sns.set_theme(style="whitegrid", context="talk")

# Build a simple baseline for presentation-oriented model comparison.
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_prob = baseline_model.predict_proba(X_test)[:, 1]

model_comparison = pd.DataFrame([
    {
        "Model": "Baseline Dummy Classifier",
        "Accuracy": accuracy_score(y_test, baseline_pred),
        "Precision": precision_score(y_test, baseline_pred, zero_division=0),
        "Recall": recall_score(y_test, baseline_pred, zero_division=0),
        "F1 Score": f1_score(y_test, baseline_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, baseline_prob),
    },
    {
        "Model": "Random Forest (Selected)",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    },
])

print("Model Comparison Table:")
display(model_comparison.round(4))

# Confusion Matrix visualization.
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Predicted No Churn", "Predicted Churn"],
    yticklabels=["Actual No Churn", "Actual Churn"],
)
plt.title("Confusion Matrix: Random Forest Churn Predictions", pad=16, weight="bold")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# ROC Curve comparison.
rf_fpr, rf_tpr, _ = roc_curve(y_test, y_prob)
baseline_fpr, baseline_tpr, _ = roc_curve(y_test, baseline_prob)

plt.figure(figsize=(9, 6))
plt.plot(rf_fpr, rf_tpr, linewidth=3, label=f"Random Forest (AUC = {roc_auc_score(y_test, y_prob):.3f})", color="#0f766e")
plt.plot(baseline_fpr, baseline_tpr, linewidth=2, linestyle="--", label=f"Baseline Dummy (AUC = {roc_auc_score(y_test, baseline_prob):.3f})", color="#94a3b8")
plt.plot([0, 1], [0, 1], linestyle=":", color="#475569", label="No-Skill Reference")
plt.title("ROC Curve: Selected Model vs Baseline", pad=16, weight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

# Precision-Recall Curve comparison.
rf_precision_curve, rf_recall_curve, _ = precision_recall_curve(y_test, y_prob)
baseline_precision_curve, baseline_recall_curve, _ = precision_recall_curve(y_test, baseline_prob)
rf_pr_auc = average_precision_score(y_test, y_prob)
baseline_pr_auc = average_precision_score(y_test, baseline_prob)

plt.figure(figsize=(9, 6))
plt.plot(rf_recall_curve, rf_precision_curve, linewidth=3, label=f"Random Forest (AP = {rf_pr_auc:.3f})", color="#b45309")
plt.plot(baseline_recall_curve, baseline_precision_curve, linewidth=2, linestyle="--", label=f"Baseline Dummy (AP = {baseline_pr_auc:.3f})", color="#94a3b8")
plt.title("Precision-Recall Curve: Selected Model vs Baseline", pad=16, weight="bold")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

# Expanded feature importance view with business-friendly interpretation.
top_feature_importance = feature_importances.nlargest(15).sort_values()
plt.figure(figsize=(12, 8))
top_feature_importance.plot(kind="barh", color=sns.color_palette("crest", n_colors=15))
plt.title("Top 15 Feature Importances for Customer Churn", pad=16, weight="bold")
plt.xlabel("Relative Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

feature_interpretation = pd.DataFrame({
    "Feature": top_feature_importance.sort_values(ascending=False).index,
    "Importance": top_feature_importance.sort_values(ascending=False).values,
})

def interpret_feature(feature_name):
    if "Tenure Months" in feature_name:
        return "Customer lifetime remains one of the strongest signals of churn risk."
    if "Monthly Charges" in feature_name or "Total Charges" in feature_name:
        return "Pricing and bill size materially influence churn behavior."
    if "Contract_" in feature_name:
        return "Contract structure shapes commitment and customer retention."
    if "Internet Service_" in feature_name:
        return "Service mix is linked to churn differences across internet plans."
    if "Payment Method_" in feature_name:
        return "Payment experience and billing habits are meaningful churn signals."
    if "Online Security" in feature_name or "Tech Support" in feature_name:
        return "Support and protection services can reduce perceived switching risk."
    return "This feature contributes to the model?s customer-risk ranking."

feature_interpretation["Business Interpretation"] = feature_interpretation["Feature"].apply(interpret_feature)
print("Top Feature Importance with Business Interpretation:")
display(feature_interpretation.round({"Importance": 4}))


## Segmentation


In [ ]:
# Build customer segments using tenure, charges, and predicted churn probability.
all_probs = rf_final.predict_proba(X)[:, 1]

segmentation_data = pd.DataFrame(
    {
        "Tenure Months": df_clean["Tenure Months"],
        "Monthly Charges": df_clean["Monthly Charges"],
        "Total Charges": df_clean["Total Charges"],
        "Churn Probability": all_probs,
    }
)

features_to_scale = ["Tenure Months", "Monthly Charges", "Total Charges", "Churn Probability"]
scaler = StandardScaler()
scaled_data = scaler.fit_transform(segmentation_data[features_to_scale])

wcss = []
for k in range(1, 16):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 6))
plt.plot(range(1, 16), wcss, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS")
plt.title("Elbow Method for Optimal K")
plt.show()

kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
segmentation_data["Cluster"] = kmeans.fit_predict(scaled_data)

cluster_names = {
    0: "Budget Loyal Customer",
    1: "High Risk New Customer",
    2: "Loyal Premium Customer",
}
segmentation_data["Cluster Segment"] = segmentation_data["Cluster"].map(cluster_names)

cluster_summary = segmentation_data.groupby("Cluster Segment").mean(numeric_only=True)
print("\nCluster summary:")
print(cluster_summary)

plt.figure(figsize=(12, 6))
sns.scatterplot(
    x="Tenure Months",
    y="Churn Probability",
    hue="Cluster Segment",
    data=segmentation_data,
    palette="viridis",
    alpha=0.6,
)
plt.title("Business Insights: Tenure vs Churn Risk by Segment")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

plt.figure(figsize=(10, 5))
sns.scatterplot(
    x="Monthly Charges",
    y="Churn Probability",
    hue="Cluster Segment",
    data=segmentation_data,
    palette="viridis",
)
plt.title("Monthly Charges vs Churn Probability by Customer Segment")
plt.xlabel("Monthly Charges")
plt.ylabel("Churn Probability")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


## Segment Reporting and Recommendations
This section summarizes the customer personas in a portfolio-ready format and links each segment to practical actions.


In [ ]:
segment_report = (
    segmentation_data.groupby("Cluster Segment")
    .agg(
        Customer_Count=("Cluster Segment", "size"),
        Average_Tenure_Months=("Tenure Months", "mean"),
        Average_Monthly_Charges=("Monthly Charges", "mean"),
        Average_Churn_Probability=("Churn Probability", "mean"),
    )
    .sort_values("Average_Churn_Probability", ascending=False)
    .round(2)
)

print("Customer Segment Summary:")
display(segment_report)

plt.figure(figsize=(10, 6))
segment_counts_plot = segment_report["Customer_Count"].sort_values(ascending=False)
ax = sns.barplot(
    x=segment_counts_plot.values,
    y=segment_counts_plot.index,
    hue=segment_counts_plot.index,
    palette="viridis",
    legend=False,
)
plt.title("Customer Count by Segment", pad=16, weight="bold")
plt.xlabel("Number of Customers")
plt.ylabel("Segment")
for index, value in enumerate(segment_counts_plot.values):
    ax.text(value + 20, index, f"{value:,}", va="center")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 6))
segment_profile_long = segment_report.reset_index().melt(
    id_vars="Cluster Segment",
    value_vars=["Average_Tenure_Months", "Average_Monthly_Charges", "Average_Churn_Probability"],
    var_name="Metric",
    value_name="Value",
)
sns.barplot(
    data=segment_profile_long,
    x="Cluster Segment",
    y="Value",
    hue="Metric",
    palette="Set2",
)
plt.title("Segment Profile Comparison", pad=16, weight="bold")
plt.xlabel("Customer Segment")
plt.ylabel("Average Value")
plt.xticks(rotation=10)
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

segment_recommendations = {
    "High Risk New Customer": [
        "Launch early-life retention offers during the first 3-6 months.",
        "Use proactive onboarding and service check-ins to reduce first-year churn.",
        "Target pricing reassurance or bundle incentives for customers with high monthly charges.",
    ],
    "Budget Loyal Customer": [
        "Protect value perception with low-cost loyalty rewards and contract renewal nudges.",
        "Promote add-on services selectively without disrupting the current price-to-value balance.",
        "Use referral and advocacy campaigns because this segment is relatively stable.",
    ],
    "Loyal Premium Customer": [
        "Prioritize white-glove support and premium service reliability communications.",
        "Offer targeted upsell bundles that deepen engagement rather than discounting broadly.",
        "Monitor for service-experience issues because these customers represent higher revenue value.",
    ],
}

print("Business Recommendations by Segment:")
for segment, recommendations in segment_recommendations.items():
    print(f"\n{segment}:")
    for recommendation in recommendations:
        print(f"- {recommendation}")


## Business Insights


In [ ]:
final_recall = classification_report(y_test, y_pred, output_dict=True)["1"]["recall"]
final_accuracy = accuracy_score(y_test, y_pred)
segment_counts = segmentation_data["Cluster Segment"].value_counts()

print("====================================================")
print("        FINAL PROJECT VALUE & PERFORMANCE           ")
print("====================================================")
print(f"[+] Model Recall:   {final_recall:.2%} (Churners Captured)")
print(f"[+] Model Accuracy: {final_accuracy:.2%} (Overall Correctness)")
print("----------------------------------------------------")
print("BUSINESS SEGMENTATION BREAKDOWN:")
for segment, count in segment_counts.items():
    percentage = (count / len(segmentation_data)) * 100
    print(f"- {segment}: {count} customers ({percentage:.1f}%)")
print("----------------------------------------------------")
print("VALUE PROPOSITION:")
print("The model effectively identifies high-risk churners,")
print("allowing the marketing team to target interventions before")
print("revenue loss occurs. The segmentation further refines this")
print("by distinguishing between 'High Risk' and 'Loyal' profiles.")
print("====================================================")

print("\nKey business insights:")
print("- Lower-tenure customers show a higher propensity to churn.")
print("- Month-to-month contracts have a significantly higher churn rate.")
print("- Higher monthly charges are associated with elevated churn risk.")
print("- Segment-specific strategies can improve retention and targeting.")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")


## Executive Summary
A concise, stakeholder-ready summary of the project objective, performance, churn drivers, and recommended actions.


In [ ]:
top_drivers = feature_importances.nlargest(5).index.tolist()
business_actions = [
    "Prioritize early-tenure customer retention plays.",
    "Reduce churn exposure in month-to-month and higher-charge customer groups.",
    "Use segment-based offers instead of one-size-fits-all retention campaigns.",
]

print("=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)
print("Project Objective:")
print("Build a churn prediction and customer segmentation workflow that helps a telecom business identify at-risk customers and target retention actions.")
print("\nDataset Size:")
print(f"{len(df_clean):,} customer records and {df_clean.shape[1]} raw columns")
print("\nBest Model:")
print("Random Forest Classifier (n_estimators=300, max_depth=10, class_weight='balanced')")
print("\nKey Metrics:")
print(f"- Accuracy:  {accuracy_score(y_test, y_pred):.2%}")
print(f"- Precision: {precision_score(y_test, y_pred):.2%}")
print(f"- Recall:    {recall_score(y_test, y_pred):.2%}")
print(f"- F1 Score:  {f1_score(y_test, y_pred):.2%}")
print(f"- ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print("\nMain Churn Drivers:")
for driver in top_drivers:
    print(f"- {driver}")
print("\nBusiness Actions:")
for action in business_actions:
    print(f"- {action}")
